In [1]:
!pip install ultralytics filterpy opencv-python pandas pyarrow huggingface_hub yt-dlp -q
!apt-get install -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


TASK 1 : Drone Object Detection

In [2]:
!yt-dlp -f "bestvideo[ext=mp4]" -o "drone_video_1.mp4" "https://www.youtube.com/watch?v=DhmZ6W1UAv4"

[youtube] Extracting URL: https://www.youtube.com/watch?v=DhmZ6W1UAv4
[youtube] DhmZ6W1UAv4: Downloading webpage
[youtube] DhmZ6W1UAv4: Downloading android vr player API JSON
[info] DhmZ6W1UAv4: Downloading 1 format(s): 137
[download] Destination: drone_video_1.mp4
[download] 100% of   10.89MiB in 00:00:00 at 38.19MiB/s


In [3]:
!yt-dlp -f "bestvideo[ext=mp4]" -o "drone_video_2.mp4" "https://www.youtube.com/watch?v=YrydHPwRelI"

[youtube] Extracting URL: https://www.youtube.com/watch?v=YrydHPwRelI
[youtube] YrydHPwRelI: Downloading webpage
[youtube] YrydHPwRelI: Downloading android vr player API JSON
[info] YrydHPwRelI: Downloading 1 format(s): 137
[download] Destination: drone_video_2.mp4
[download] 100% of   37.23MiB in 00:00:02 at 17.45MiB/s


In [4]:
!mkdir -p frames_1 frames_2
!ffmpeg -i drone_video_1.mp4 -vf "fps=5" frames_1/frame_%04d.jpg -y
!ffmpeg -i drone_video_2.mp4 -vf "fps=5" frames_2/frame_%04d.jpg -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [19]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

model_path = hf_hub_download(
    repo_id="mshamrai/yolov8n-visdrone",
    filename="best.pt"
)
model = YOLO(model_path)
print("Model loaded!", model.ckpt_path)


Model loaded! /root/.cache/huggingface/hub/models--mshamrai--yolov8n-visdrone/snapshots/28ca5101e5e6090c6beba15eb79d038d0befad6d/best.pt


In [25]:
import os
import cv2
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from huggingface_hub import login, hf_hub_download

#config
FRAME_DIRS = ["frames_1","frames_2"]
DETECTIONS_DIR = "detections"
CONF_THRESH = 0.25
MIN_SEG_LENGTH = 3
GAP_ALLOWED = 2

os.makedirs(DETECTIONS_DIR, exist_ok=True)
all_records = []

def get_segments(frame_indices, gap_allowed):
  # this function groups frame indices into segments and tolerates small gaps
  if not frame_indices:
    return[]
  segments = []
  current = [frame_indices[0]]
  for i in range(1, len(frame_indices)):
    if frame_indices[i]-frame_indices[i-1] <= gap_allowed + 1:
      current.append(frame_indices[i])
    else:
      segments.append(current)
      current = [frame_indices[i]]
  segments.append(current)
  return segments

for frame_dir in FRAME_DIRS:
  frames = sorted(Path(frame_dir).glob("*.jpg"))

  #running detection on all frames
  detected = {}
  for i, fp in enumerate(frames):
    results = model(str(fp), conf = CONF_THRESH, verbose = False)[0]
    boxes = results.boxes
    if boxes is not None and len(boxes) > 0:
      detected[i] = {"path":fp,
                     "boxes":boxes.xyxy.cpu().numpy(),
                     "confs":boxes.conf.cpu().numpy()}
  print(f"Raw Detections: {len(detected)} frames")
  #drop isolated detections
  indices = sorted(detected.keys())
  segments = get_segments(indices,GAP_ALLOWED)


  valid_segments = [s for s in segments if len(s) >=  MIN_SEG_LENGTH]
  valid_indices = set(i for seg in valid_segments for i in seg)

  print(f"  After segment filter: {len(valid_indices)} frames across {len(valid_segments)} segments")

#save valid frames

for i in valid_indices:
  if i not in detected:
    continue
  info = detected[i]
  out_name = f"{frame_dir}_{info['path'].name}"
  out_path = os.path.join(DETECTIONS_DIR, out_name)

  img = cv2.imread(str(info["path"]))
  cv2.imwrite(out_path, img)

  for box, conf in zip(info["boxes"], info["confs"]):
    all_records.append({
        "video" : frame_dir,
        "frame" : info["path"].name,
        "frame_idx" : i,
        "x1" : float(box[0]), "y1" : float(box[1]),
        "x2" : float(box[2]), "y2" : float(box[3]),
        "conf" : float(conf),
        "cx" : float((box[0] + box[2])/2),
        "cy" : float((box[1] + box[3])/2)})

df = pd.DataFrame(all_records)
df.to_parquet("detections.parquet", index = False)
print("DONE")



Raw Detections: 364 frames
  After segment filter: 354 frames across 8 segments
Raw Detections: 45 frames
  After segment filter: 26 frames across 5 segments
DONE


In [26]:
import os
import pandas as pd

#detections folder
files = os.listdir("detections")
print(f"Frames saved: {len(files)}")

# parquet
df = pd.read_parquet("detections.parquet")
print(df.head())
print(df.shape)

Frames saved: 26
      video           frame  frame_idx           x1          y1           x2  \
0  frames_2  frame_2562.jpg       2561  1606.297852  926.601013  1776.070312   
1  frames_2  frame_2563.jpg       2562  1606.480225  926.706177  1776.205078   
2  frames_2  frame_2558.jpg       2557  1609.749756  925.060913  1777.526367   
3  frames_2  frame_2559.jpg       2558  1609.519775  924.878357  1777.415039   
4  frames_2  frame_1144.jpg       1143  1601.531006  911.307251  1777.032959   

            y2      conf           cx           cy  
0  1075.508301  0.376918  1691.184082  1001.054688  
1  1075.526611  0.377590  1691.342651  1001.116394  
2  1075.384521  0.360877  1693.638062  1000.222717  
3  1075.416016  0.366212  1693.467407  1000.147217  
4  1066.710327  0.568599  1689.281982   989.008789  
(26, 10)


TASK 2: Kalman filter Tracking

In [34]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from filterpy.kalman import KalmanFilter

#config
FRAME_DIRS = ["frames_2"]
OUTPUT_DIR = "output_videos"
DETECTIONS_DF = "detections.parquet"
MAX_MISSED = 5
FPS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

if len(existing[existing["video"] == "frames_2"]) < 100:
    detected2 = {}
    for i, fp in enumerate(sorted(Path("frames_2").glob("*.jpg"))):
        results = model(str(fp), conf=0.01, verbose=False)[0]
        boxes = results.boxes
        if boxes is not None and len(boxes) > 0:
            detected2[i] = {"path": fp, "boxes": boxes.xyxy.cpu().numpy(), "confs": boxes.conf.cpu().numpy()}
    print(f"  Raw: {len(detected2)} frames")
    records2 = []
    for i, info in detected2.items():
        for box, conf in zip(info["boxes"], info["confs"]):
            records2.append({"video": "frames_2", "frame": info["path"].name, "frame_idx": i,
                "x1": float(box[0]), "y1": float(box[1]), "x2": float(box[2]), "y2": float(box[3]),
                "conf": float(conf), "cx": float((box[0]+box[2])/2), "cy": float((box[1]+box[3])/2)})
    existing = existing[existing["video"] != "frames_2"]  # remove old bad data
    existing = pd.concat([existing, pd.DataFrame(records2)], ignore_index=True)
    existing.to_parquet("detections.parquet", index=False)

df = pd.read_parquet(DETECTIONS_DF)

def make_kalman():
  kf = KalmanFilter(dim_x=4, dim_z=2)
  dt = 1.0
  #state transition
  kf.F = np.array([[1,0,dt,0],
                   [0,1,0,dt],
                   [0,0,1,0],
                   [0,0,0,1]], dtype = float)
  #measurement matrix
  kf.H = np.array([[1,0,0,0],[0,1,0,0]], dtype=float)
  kf.R  *= 10    # measurement noise
  kf.P  *= 100   # initial uncertainty
  kf.Q  *= 0.1   # process noise
  return kf

def nearest_detection(pred_x, pred_y, dets, threshold=150):
    best, best_dist = None, threshold
    for det in dets:
        d = np.sqrt((det[0]-pred_x)**2 + (det[1]-pred_y)**2)
        if d < best_dist:
            best, best_dist = det, d
    return best

for frame_dir in FRAME_DIRS:
  frames = sorted(Path(frame_dir).glob("*.jpg"))
  if not frames:
    continue
  vid_df = df[df["video"] == frame_dir]
  det_by_frame = {}
  for idx, row in vid_df.iterrows():
    fi = int(row["frame_idx"])
    if fi not in det_by_frame:
      det_by_frame[fi] = []
    det_by_frame[fi].append((row["cx"], row["cy"], row["x1"], row["y1"], row["x2"], row["y2"]))


  kf = make_kalman()
  trajectory = []
  missed = 0
  initialized = False
  track_frames = []

  for i,fp in enumerate(frames):
    dets = det_by_frame.get(i,[])

    if not initialized:
      if dets:
        cx,cy = dets[0][0], dets[0][1]
        kf.x = np.array([[cx],[cy],[0],[0]], dtype = float)
        initialized = True
        trajectory.append((cx,cy))
        track_frames.append((fp, dets[0][2:], (cx,cy)))
      continue

    #predict
    kf.predict()
    pred_x, pred_y = float(kf.x[0]), float(kf.x[1])

    #update
    matched = nearest_detection(pred_x, pred_y, dets)
    if matched:
      kf.update(np.array([[matched[0]], [matched[1]]], dtype=float))
      cx, cy = float(kf.x[0]), float(kf.x[1])
      missed = 0
      trajectory.append((cx, cy))
      track_frames.append((fp, matched[2:], (cx, cy)))
    else:
      missed += 1
      if missed <= MAX_MISSED:
        trajectory.append((pred_x, pred_y))
        track_frames.append((fp, None, (pred_x, pred_y)))
      else:
        continue
  print(f"track_frames length: {len(track_frames)}")
  print(f"det_by_frame keys (first 5): {list(det_by_frame.keys())[:5]}")
  print(f"vid_df shape: {vid_df.shape}")
  #compose output video
  sample = cv2.imread(str(track_frames[0][0]))
  h, w = sample.shape[:2]
  out_path = os.path.join(OUTPUT_DIR, f"{frame_dir}_tracked.mp4")
  writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))

  for fp,box,center in track_frames:
    frame = cv2.imread(str(fp))

    #trajectory polyline
    pts = np.array(trajectory[:track_frames.index((fp,box,center))+1], dtype = np.int32)
    if len(pts) >= 2:
          cv2.polylines(frame, [pts.reshape(-1,1,2)], False, (0,255,0), 2)

    #bounding box
    if box is not None:
      x1,y1,x2,y2 = int(box[0]),int(box[1]),int(box[2]),int(box[3])
      cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 2)
      cv2.putText(frame, "drone", (x1, y1-8),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    #tracker center
    cx,cy = int(center[0]), int(center[1])
    cv2.circle(frame, (cx,cy),5, (255,0,0), -1)

    writer.write(frame)
  writer.release()
  print(f"saved {out_path}")




  Raw: 1602 frames


/tmp/ipykernel_456/735382249.py:93: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred_x, pred_y = float(kf.x[0]), float(kf.x[1])
/tmp/ipykernel_456/735382249.py:99: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  cx, cy = float(kf.x[0]), float(kf.x[1])


track_frames length: 36
det_by_frame keys (first 5): [0, 1, 2, 3, 4]
vid_df shape: (2805, 10)
saved output_videos/frames_2_tracked.mp4


In [23]:
import pandas as pd
df = pd.read_parquet("detections.parquet")
print(df["video"].unique())
print(df.shape)

['frames_2']
(26, 10)


In [30]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from filterpy.kalman import KalmanFilter

FRAME_DIRS    = ["frames_1"]
OUTPUT_DIR    = "output_videos"
MAX_MISSED    = 5
FPS           = 5

existing = pd.read_parquet("detections.parquet")
if "frames_1" not in existing["video"].values:
    print("Detecting frames_1...")
    detected = {}
    for i, fp in enumerate(sorted(Path("frames_1").glob("*.jpg"))):
        results = model(str(fp), conf=0.01, verbose=False)[0]
        boxes = results.boxes
        if boxes is not None and len(boxes) > 0:
            detected[i] = {"path": fp, "boxes": boxes.xyxy.cpu().numpy(), "confs": boxes.conf.cpu().numpy()}
    print(f"  Raw: {len(detected)} frames")
    records = []
    for i, info in detected.items():
        for box, conf in zip(info["boxes"], info["confs"]):
            records.append({"video": "frames_1", "frame": info["path"].name, "frame_idx": i,
                "x1": float(box[0]), "y1": float(box[1]), "x2": float(box[2]), "y2": float(box[3]),
                "conf": float(conf), "cx": float((box[0]+box[2])/2), "cy": float((box[1]+box[3])/2)})
    existing = pd.concat([existing, pd.DataFrame(records)], ignore_index=True)
    existing.to_parquet("detections.parquet", index=False)
    print(f" Added {len(records)} detections for frames_1")

df = pd.read_parquet("detections.parquet")

def make_kalman():
    kf = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0
    kf.F = np.array([[1,0,dt,0],
                     [0,1,0,dt],
                     [0,0,1, 0],
                     [0,0,0, 1]], dtype=float)
    kf.H = np.array([[1,0,0,0],
                     [0,1,0,0]], dtype=float)
    kf.R *= 10
    kf.P *= 100
    kf.Q *= 0.1
    return kf

def nearest_detection(pred_x, pred_y, dets, threshold=150):
    best, best_dist = None, threshold
    for det in dets:
        d = np.sqrt((det[0]-pred_x)**2 + (det[1]-pred_y)**2)
        if d < best_dist:
            best, best_dist = det, d
    return best

for frame_dir in FRAME_DIRS:
    frames = sorted(Path(frame_dir).glob("*.jpg"))
    if not frames:
        continue
    print(f"Tracking {frame_dir} — {len(frames)} frames")

    vid_df = df[df["video"] == frame_dir]
    det_by_frame = {}
    for idx, row in vid_df.iterrows():
        fi = int(row["frame_idx"])
        if fi not in det_by_frame:
            det_by_frame[fi] = []
        det_by_frame[fi].append((row["cx"], row["cy"], row["x1"], row["y1"], row["x2"], row["y2"]))

    kf = make_kalman()
    trajectory = []
    missed = 0
    initialized = False
    track_frames = []

    for i, fp in enumerate(frames):
        dets = det_by_frame.get(i, [])
        if not initialized:
            if dets:
                cx, cy = dets[0][0], dets[0][1]
                kf.x = np.array([[cx],[cy],[0],[0]], dtype=float)
                initialized = True
                trajectory.append((cx, cy))
                track_frames.append((fp, dets[0][2:], (cx, cy)))
            continue

        kf.predict()
        pred_x, pred_y = float(kf.x[0][0]), float(kf.x[1][0])

        matched = nearest_detection(pred_x, pred_y, dets)
        if matched:
            kf.update(np.array([[matched[0]], [matched[1]]], dtype=float))
            cx, cy = float(kf.x[0][0]), float(kf.x[1][0])
            missed = 0
            trajectory.append((cx, cy))
            track_frames.append((fp, matched[2:], (cx, cy)))
        else:
            missed += 1
            if missed <= MAX_MISSED:
                trajectory.append((pred_x, pred_y))
                track_frames.append((fp, None, (pred_x, pred_y)))

    print(f"  Tracked {len(track_frames)} frames")
    sample = cv2.imread(str(track_frames[0][0]))
    h, w = sample.shape[:2]
    out_path = os.path.join(OUTPUT_DIR, f"{frame_dir}_tracked.mp4")
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))

    for j, (fp, box, center) in enumerate(track_frames):
        frame = cv2.imread(str(fp))
        pts = np.array(trajectory[:j+1], dtype=np.int32)
        if len(pts) >= 2:
            cv2.polylines(frame, [pts.reshape(-1,1,2)], False, (0,255,0), 2)
        if box is not None:
            x1,y1,x2,y2 = int(box[0]),int(box[1]),int(box[2]),int(box[3])
            cv2.rectangle(frame, (x1,y1),(x2,y2),(0,0,255),2)
            cv2.putText(frame, "drone", (x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
        cx, cy = int(center[0]), int(center[1])
        cv2.circle(frame, (cx,cy), 5, (255,0,0), -1)
        writer.write(frame)

    writer.release()
    print(f" Saved {out_path}")

Detecting frames_1...
  Raw: 825 frames
 Added 2643 detections for frames_1
Tracking frames_1 — 828 frames
  Tracked 253 frames
 Saved output_videos/frames_1_tracked.mp4


In [35]:
import os
print(os.listdir("output_videos"))

['frames_2_tracked.mp4', 'frames_1_tracked.mp4']


In [ ]:


from huggingface_hub import HfApi
api = HfApi()

# delete old repo
api.delete_repo(
    repo_id="sania808/drone-detections",
    repo_type="dataset",
    token="secret"
)

# recreate and upload fresh
api.create_repo(
    repo_id="sania808/drone-detections",
    repo_type="dataset",
    token="secret"
)

api.upload_file(
    path_or_fileobj="detections.parquet",
    path_in_repo="detections.parquet",
    repo_id="sania808/drone-detections",
    repo_type="dataset",
    token="secret"
)
print(" View at: https://huggingface.co/datasets/sania808/drone-detections")